# PathEval: full pipeline (Kaggle)

Runs the team's grounding, prompting, image-only scaffolding, and evaluation modules.
See the root README for ownership, methods, limitations, and 15-example settings.

**Settings:** GPU T4 x2, Internet on.
**Input:** optional prepared test dataset; downloaded directly if absent.
**Secrets:** no GitHub token required for the public repository; HF_TOKEN is optional.
Use a new OUT_DIR when code, data, or settings change. Download it before stopping.


## 1. Settings

In [ ]:
REPO = "zoekrishtul/Deep-Learning-Final-Project"
BRANCH = "main"
CONDITIONS = [                       # see src/pipeline.py for what each one means
    "direct", "step_by_step", "structured",          # prompting only
    "direct+scaffold", "structured+scaffold",        # visual scaffolding
    "direct+clip", "structured+clip",                # learned grounding (+ reasoning)
    "structured+oracle",                             # upper bound: true attributes
]
RUN_SWAPPED = True                   # also run every condition with left/right swapped
N_PER_SCENARIO = 10                  # 10 -> 150 test samples; None -> all 1050
N_TRAIN = 500                        # grounding training records (Yu-Jye's notebook used 500)
BATCH_SIZE = 8
OUT_DIR = "/kaggle/working/results"

## 2. Get the code (token never printed; safe to rerun)

In [ ]:
import os, sys, subprocess
os.chdir("/kaggle/working")
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

def secret(name):
    try:
        return secrets.get_secret(name) if secrets else None
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")

def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        msg = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git {args[0]} failed:\n{msg}")
    return result.stdout

auth = f"x-access-token:{token}@" if token else ""
subprocess.run(["rm", "-rf", "/tmp/repo"], check=True)
git("clone", "-q", "--branch", BRANCH, f"https://{auth}github.com/{REPO}.git", "/tmp/repo")
git("-C", "/tmp/repo", "remote", "set-url", "origin", f"https://github.com/{REPO}.git")
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")
print(git("log", "--oneline", "-1"))

### Install dependencies
Run after cloning, before importing project modules. If Kaggle requests a restart, restart and rerun Settings and Get the code before continuing.

In [ ]:
%pip install -r /tmp/repo/requirements.txt

## 3. Test data (attached input or direct download)

In [ ]:
import glob, json
from src.data import (load_manifest, manifest_from_clean, write_jsonl,
                      stratified_subset, fetch_patheval)

found = (glob.glob("/kaggle/input/**/test_2D.jsonl", recursive=True) or
         glob.glob("/kaggle/input/**/test_2D_clean.json", recursive=True))
if found:
    DATA_ROOT = os.path.dirname(found[0])
    if found[0].endswith(".jsonl"):
        MANIFEST = found[0]
    else:
        MANIFEST = "/kaggle/working/test_2D.jsonl"
        with open(found[0]) as f:
            write_jsonl(manifest_from_clean(json.load(f)), MANIFEST)
else:
    DATA_ROOT = "/kaggle/working/patheval_test"
    MANIFEST = fetch_patheval("test", DATA_ROOT)

test_rows = load_manifest(MANIFEST)
samples = stratified_subset(test_rows, N_PER_SCENARIO) if N_PER_SCENARIO else test_rows
print(len(test_rows), "test samples;", len(samples), "used | data root:", DATA_ROOT)


## 4. Grounding: train on PathEval train images, predict for test samples
Skipped if no condition uses +clip. First-time data/model downloads may take several minutes.

In [ ]:
import torch
ATTRIBUTES_PATH = f"{OUT_DIR}/attributes_test.jsonl"
if any(c.endswith("+clip") for c in CONDITIONS):
    from src.data import fetch_patheval
    from src.grounding import (ClipPairEncoder, train_grounding, save_grounding,
                               predict_attributes)
    TRAIN_ROOT = "/kaggle/working/patheval_train"
    train_rows = load_manifest(fetch_patheval("train", TRAIN_ROOT, limit=N_TRAIN))

    encoder = ClipPairEncoder()
    bundle = train_grounding(train_rows, TRAIN_ROOT, encoder)
    os.makedirs(OUT_DIR, exist_ok=True)
    save_grounding(bundle, f"{OUT_DIR}/grounding.pt")
    with open(f"{OUT_DIR}/grounding_comparison.json", "w") as f:
        json.dump(bundle["comparison"], f, indent=2)

    write_jsonl(predict_attributes(samples, DATA_ROOT, bundle, encoder), ATTRIBUTES_PATH)
    print("Attribute predictions ->", ATTRIBUTES_PATH)
    del encoder, bundle
    torch.cuda.empty_cache()             # free GPU memory for Qwen

## 5. Load Qwen2-VL-7B (float16, split across both T4s)

In [ ]:
from src.prompting import QwenPathEvaluator
evaluator = QwenPathEvaluator()
evaluator.describe()

## 6. Run all conditions
Resume only with unchanged code, data, and settings. Use a new OUT_DIR otherwise.
Reasoning conditions take longer than direct; progress estimates the remaining time.

In [ ]:
from src.pipeline import run_conditions
outputs = run_conditions(CONDITIONS, MANIFEST, DATA_ROOT, OUT_DIR, evaluator,
                         swaps=(False, True) if RUN_SWAPPED else (False,),
                         attributes_path=ATTRIBUTES_PATH, n_per_scenario=N_PER_SCENARIO,
                         batch_size=BATCH_SIZE)

## 7. Results table
Accuracy and invalid rate come from Zoe's `evaluate()`. *consistent* = picks the same underlying path
when left/right are swapped. *+/- vs direct* = samples this condition gets right that direct gets wrong,
and vice versa; *p* = exact McNemar test.

In [ ]:
from src.pipeline import compare, format_table, save_table
rows = compare(OUT_DIR, CONDITIONS)
print(format_table(rows))
save_table(rows, f"{OUT_DIR}/comparison.csv")

Download the entire folder specified by OUT_DIR from the Output panel before stopping. Preserve predictions, configs, comparison.csv, grounding metrics, and the checkpoint. The default is /kaggle/working/results/; a small test may use another folder.